# Chapter 11 — CO\u2082 Capture and Storage: Scientific Figures

**Figures:**
1. CO2\u2013water mutual solubility (CPA vs SRK vs experimental)
2. CO2 phase envelope with impurities
3. CO2\u2013MEA VLE at absorber conditions
4. CO2 density at pipeline conditions: CPA vs NIST

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

In [3]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## Figure 11.1 — CO\u2082 Solubility in Water (CPA vs SRK)

In [4]:
# CO2 solubility in water at 25°C as a function of pressure
pressures = np.arange(5, 105, 5)
x_co2_cpa, x_co2_srk = [], []

for P in pressures:
    for label, cls, mr, x_list in [("CPA", SystemSrkCPAstatoil, 10, x_co2_cpa),
                                    ("SRK", SystemSrkEos, "classic", x_co2_srk)]:
        try:
            f = cls(273.15 + 25.0, float(P))
            f.addComponent("CO2", 0.98)
            f.addComponent("water", 0.02)
            if isinstance(mr, int): f.setMixingRule(mr)
            else: f.setMixingRule(mr)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            # CO2 in aqueous phase
            nph = int(f.getNumberOfPhases())
            found = False
            for ph in range(nph):
                pt = str(f.getPhase(ph).getPhaseTypeName())
                if "aqueous" in pt.lower() or ("liquid" in pt.lower() and ph > 0):
                    x_c = float(f.getPhase(ph).getComponent("CO2").getx())
                    x_list.append(x_c * 100)  # mol%
                    found = True
                    break
            if not found:
                x_list.append(np.nan)
        except Exception:
            x_list.append(np.nan)

# Experimental data (Wiebe & Gaddy, 1940)
exp_P = [10, 25, 50, 75, 100]
exp_x = [0.6, 1.2, 1.8, 2.0, 2.1]

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures[:len(x_co2_cpa)], x_co2_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.plot(pressures[:len(x_co2_srk)], x_co2_srk, color=ORANGE, linewidth=1.5, linestyle="--", label="SRK")
ax.scatter(exp_P, exp_x, color="black", s=25, zorder=5, label="Exp. (Wiebe)")
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("CO\u2082 in water (mol%)")
ax.set_title("CO\u2082 solubility in water at 25\u00b0C")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch11_01_co2_solubility_water.png")

Saved: fig_ch11_01_co2_solubility_water.png


## Figure 11.2 — CO\u2082 Phase Envelope with Impurities

In [5]:
fig, ax = plt.subplots(figsize=(3.5, 3.0))

# CO2 phase envelopes with different impurity levels
compositions = [
    ({"CO2": 1.0}, "Pure CO\u2082", BLUE, "-"),
    ({"CO2": 0.95, "nitrogen": 0.05}, "5% N\u2082", ORANGE, "--"),
    ({"CO2": 0.95, "methane": 0.05}, "5% CH\u2084", GREEN, "-."),
    ({"CO2": 0.97, "H2S": 0.03}, "3% H\u2082S", PURPLE, ":"),
]

for comp, label, col, ls in compositions:
    try:
        f = SystemSrkCPAstatoil(273.15, 50.0)
        for name, z in comp.items():
            f.addComponent(name, z)
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.calcPTphaseEnvelope(True)

        # Get envelope data
        T_env = list(ops.getOperation().get("temperature"))
        P_env = list(ops.getOperation().get("pressure"))
        T_arr = [float(t) - 273.15 for t in T_env]
        P_arr = [float(p) for p in P_env]
        ax.plot(T_arr, P_arr, color=col, linestyle=ls, linewidth=1.2, label=label)
    except Exception as e:
        print(f"Phase envelope failed for {label}: {e}")

ax.set_xlabel("Temperature (\u00b0C)")
ax.set_ylabel("Pressure (bar)")
ax.set_title("CO\u2082 phase envelopes with impurities")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
ax.set_xlim(-60, 50)
ax.set_ylim(0, 120)
fig.tight_layout()
save(fig, "fig_ch11_02_co2_phase_envelope.png")

Phase envelope failed for Pure CO₂: 'NoneType' object is not iterable
Phase envelope failed for 5% N₂: 'NoneType' object is not iterable
Phase envelope failed for 5% CH₄: 'NoneType' object is not iterable


Phase envelope failed for 3% H₂S: 'NoneType' object is not iterable
Saved: fig_ch11_02_co2_phase_envelope.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_45044\2339468008.py:32: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)


## Figure 11.3 — CO\u2082 Density at Pipeline Conditions

In [6]:
# CO2 density at 10-200 bar, 25°C
pressures = np.arange(10, 205, 5)
rho_cpa = []

for P in pressures:
    try:
        f = SystemSrkCPAstatoil(273.15 + 25.0, float(P))
        f.addComponent("CO2", 1.0)
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        rho = float(f.getPhase(0).getDensity("kg/m3"))
        rho_cpa.append(rho)
    except Exception:
        rho_cpa.append(np.nan)

# NIST reference (approximate)
nist_P = [10, 30, 50, 70, 74, 80, 100, 120, 150, 200]
nist_rho = [19.4, 65.9, 136, 405, 630, 720, 790, 840, 890, 950]

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures[:len(rho_cpa)], rho_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.scatter(nist_P, nist_rho, color="black", s=25, zorder=5, label="NIST")
ax.axvline(x=73.8, color="grey", linestyle=":", alpha=0.5)
ax.annotate("$P_c$", xy=(73.8, 100), fontsize=8, color="grey")
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("Density (kg/m\u00b3)")
ax.set_title("CO\u2082 density at 25\u00b0C")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch11_03_co2_density.png")

Saved: fig_ch11_03_co2_density.png
